# Cvičení 1

### Řetězce

Typ `Char` už známe.

In [1]:
'a'

'a'

In [2]:
:t 'a'

'a' :: Char

Pracovat často budeme s textovými řetězci:

In [3]:
"ahoj"

"ahoj"

Už ale víme, že řetězec je ve skutečnosti jen seznam znaků:

In [4]:
[ 'a', 'h', 'o', 'j' ]

"ahoj"

In [5]:
myStr = "ahoj"

In [6]:
:t myStr

myStr :: String

In [7]:
myStr2 = ['a', 'h', 'o', 'j']

In [8]:
:t myStr2

myStr2 :: [Char]

In [9]:
:i String

type String :: *
type String = [Char]
  	-- Defined in ‘GHC.Base’

In [10]:
myStr == myStr2

True

GHCi a Jupyter se vždycky snaží vypsat každý vyhodnocený výraz, ale to je jen věc interaktivního prostředí.\
Když něco budeme chtít vypisovat ve „skutečných programech“, budeme k tomu potřebovat nějakou „vstupně-výstupní akci“ – reprezentovanou třeba funkcí `print`.

In [11]:
print ['a', 'h', 'o', 'j']

"ahoj"

Zobrazte si typovou signaturu funkce `print`.

In [12]:
:t print

print :: forall a. Show a => a -> IO ()

Dá se to číst jako „vezme to věc jakéhokoliv typu `a`, která podporuje jakési „rozhraní“ (typovou třídu) `Show`, a vrátí to nějakou speciální _věc_ reprezentující I/O akci, která už sama nevrací žádný smysluplný výsledek“.\
Tuhle I/O akci pak vezme interpret a spustí ji – což provede výpis na výstup.\
Tohle je takový sneak peek na princip, jakým může funkcionální svět interagovat se zlým *non-pure* světem, který má vedlejší efekty (jako třeba výpis něčeho do terminálu). Později se tím budeme hodně zabývat.

Podobnou funkcí je `putStrLn`. Všimněte si ale jiné typové signatury.

In [13]:
putStrLn "ahoj"

ahoj

In [14]:
:t putStrLn

putStrLn :: String -> IO ()

**Otázka:** Budou následující tři výrazy fungovat? Pokud ne, proč? Jak to ověřit?

In [15]:
print 8943

8943

In [16]:
putStrLn 8943

: 

In [17]:
putStrLn (show 8943)

Line 1: Use print
Found:
putStrLn (show 8943)
Why not:
print 8943

8943

**Otázka:** Jak se dá předchozí výraz přepsat, aby tam nebyly závorky?

### Základní práce se seznamy

Několik základních funkcí pro práci s jakýmikoliv seznamy. Začneme konkatenací:

In [18]:
myStr :: String  -- zvykejme si anotovat všechny funkce (i konstantní ne-funkce bez parametrů) typovou anotací
myStr = "hello " ++ "world"

In [19]:
myStr

"hello world"

In [20]:
myStr = concat [ "hello ", "world" ]

Line 1: Use ++
Found:
concat ["hello ", "world"]
Why not:
"hello " ++ "world"

In [21]:
myStr

"hello world"

In [22]:
myStr = concat [ "jag ", [ 'm', 'å', 'r' ], " bra" ]
myStr

"jag m\229r bra"

In [23]:
putStrLn myStr

jag mår bra

In [24]:
:t (++)

(++) :: forall a. [a] -> [a] -> [a]

**Otázka:** Jaká bude typová anotace `concat`?

In [25]:
:t concat
-- concat :: forall a. ?

concat :: forall (t :: * -> *) a. Foldable t => t [a] -> [a]

Jde to i pro seznamy jiných typů:

In [26]:
[1, 2, 3] ++ [4, 5, 6]

[1,2,3,4,5,6]

**Otázka:** Jaká bude typová anotace následujícího výrazu?

In [27]:
:t (++ myStr)

(++ myStr) :: [Char] -> [Char]

Na základě typové anotace by mělo být úplně jasné, jaký bude výsledek zde:

In [28]:
(++ myStr) [1,2]  -- [1,2] ++ myStr

: 

In [29]:
l :: [Int]
l = [4, 2, 3, 6, 9, 2, 3, 1, 2, 3]

`head` vrací první prvek seznamu, často se mu říká **hlavička**:

In [30]:
head l

4

**Otázka:** Co se stane při vyhodnocování dalšího výrazu?

In [31]:
head []

: 

`tail` vrací zbytek po odstranění hlavičky:

In [32]:
tail l

[2,3,6,9,2,3,1,2,3]

Prvních $n$ prvků seznamu:

In [33]:
take 3 l

[4,2,3]

**Otázka:** Jaká je typová anotace `take`?

In [34]:
:t take

take :: forall a. Int -> [a] -> [a]

Všechno kromě prvních $n$ prvků seznamu:

In [35]:
drop 1 l

[2,3,6,9,2,3,1,2,3]

Operátor pro „indexaci“ (od nuly):

In [36]:
l !! 2

3

**Otázka:** Jaká je typová anotace operátoru `!!`?

In [37]:
:t (!!)

(!!) :: forall a. HasCallStack => [a] -> Int -> a

Délka seznamu:

In [38]:
length l

10

**Cvičení:** (Bez vyhodnocování) přiřaďte výrazy k odpovídající redukované formě:
1. `"Jules"`
2. `[2,3,5,6,8,9]`
3. `"rainbow"`
4. `[6,12,18]`
5. `10`

In [39]:
-- a)
concat [[1 * 6], [2 * 6], [3 * 6]]
-- b)
"rain" ++ drop 2 "elbow"
-- c)
10 * head [1, 2, 3]
-- d)
take 3 "Julie" ++ tail "yes"
-- e)
concat [tail [1, 2, 3],
        tail [4, 5, 6],
        tail [7, 8, 9]]

[6,12,18]

"rainbow"

10

"Jules"

[2,3,5,6,8,9]

**Na tomto místě si vypracujte cvičení 1 až 2 z odevzdávaného notebooku.**

## Datové typy
Typy jsou fajn :) Postupně zjistíme, jak nám pomáhají elegantně zobecňovat řešení různých problémů.

Typ vzniká pomocí deklarace `data`. Datová deklarace se skládá z **typového konstruktoru** a **datových konstruktorů**.\
Typový konstruktor zatím považujme v podstatě za název typu (později zjistíme, proč se tomu říká *konstruktor*).\
Datové konstruktory jsou v podstatě možné **hodnoty** typu. Proč konstruktory? Umožňují nám totiž (někde v paměti) vytvořit instanci toho typu! Ony to jsou ve skutečnosti tak trochu *funkce*. :)

Takhle může vypadat deklarace typu o dvou hodnotách (variace na boolean):

In [40]:
data MyBool = MyFalse | MyTrue
--   │        └ dva datové konstruktory (dvě možné hodnoty) oddělené znakem | (symbolizuje „nebo“)
--   └ typový konstruktor (název typu + kouzla)

In [41]:
MyFalse

: 

Děsivá chyba říká, že GHCi (na pozadí Jupyter jádra) neví, jak by měla na výstup vypsat hodnotu typu `MyBool`, protože tento typ není _instancí typové třídy_ `Show`.\
Naštěstí můžeme použít compiler magic, která nám to zajistí:

In [42]:
data MyBool = MyFalse | MyTrue
    deriving Show

In [43]:
MyFalse

MyFalse

Uvědomte si, kde se v kódu pracuje s čím. Podívejme se třeba na typovou signaturu funkce `not`:

In [44]:
:t not

not :: Bool -> Bool

Vidíte, že když se bavíme o *typech* a zkoumáme *typové signatury*, používáme *typové konstruktory*.

Když s funkcemi nějak pracujeme, v podstatě manipulujeme s daty, a tak používáme *datové konstruktory*:

In [45]:
not True

Line 1: Evaluate
Found:
not True
Why not:
False

False

Jak by mohla vypadat taková implementace funkce `not` pro náš vlastní typ `MyBool`?

In [46]:
not' :: MyBool -> MyBool

not' MyFalse = MyTrue
not' MyTrue = MyFalse

Tady jsme poprvé viděli, jak se píšou funkce, které reagují na konkrétní instanci dat na vstupu. Odpovídá to matematickému zápisu:
$$
not'(x) = \begin{cases}
\top & \text{if } x = \bot \\
\bot & \text{if } x = \top
\end{cases}
$$

In [47]:
not' MyTrue

MyFalse

In [48]:
not' MyFalse

MyTrue

### Číselné typy

Čísla jsou tricky. Existují různé datové typy pro celá čísla, desetinná čísla, speciální typ pro racionální čísla...

Základní dva typy pro celá čísla jsou `Int` a `Integer`. První z nich je závislý na architektuře, druhý je neomezený.

In [49]:
5 :: Int
5 :: Integer

5

5

In [50]:
12345678979876854321652487987 :: Int


<interactive>:1:1: warning: [GHC-97441] [-Woverflowed-literals] Literal 12345678979876854321652487987 is out of the Int range -9223372036854775808..9223372036854775807
-8493700344893089997

In [51]:
12345678979876854321652487987 :: Integer

12345678979876854321652487987

Když ale zkusíme zjistit typ čísla nebo nějakého výrazu s čísly pomocí `:type`/`:t`, nevidíme ani `Integer`, ani `Int`:

In [52]:
:t 2

2 :: forall {a}. Num a => a

In [53]:
:t 12345674894561657685456415685

12345674894561657685456415685 :: forall {a}. Num a => a

In [54]:
:t 45897897451566456 + 489798789798978463

45897897451566456 + 489798789798978463 :: forall {a}. Num a => a

Víme, že funkce `(+)` by asi měla umět sečíst jak dva `Int`y, tak dva `Integer`y, ale i třeba dva `Float`y.\
Zároveň víme, že `Int` a `Float` musí být jiný typ, protože se s tím interně pracuje jinak.\
Proto zavedeme různé úrovně *zobecnění* čehokoliv, čemu se dá říkat „číslo“ – a funkci `(+)` popíšeme tak, že může vzít dvě data jakéhokoliv typu, který je *instancí* `Num`, a vyprodukovat hodnotu stejného typu.

In [55]:
:t (+)

(+) :: forall a. Num a => a -> a -> a

Překladač pak uděláme dostatečně chytrý na to, aby číselnou konstantu (třeba `2`) nepovažoval za žádný konkrétní číselný typ, až dokud to nebude naprosto nevyhnutelně potřebovat k jeho vyhodnocení.\
Proto pro `:t 2` zahlásí, že `2 :: Num a => a`. Určitě to je _nějaké_ číslo, ale v tomto kontextu nepotřebuje vědět, co to bude konkrétně za typ.

Prostudujme, jaké všechny typové třídy Haskell implementuje pro `Int` a pro `Integer`. Vidíme rozdíl?

In [56]:
:i Int
:i Integer

type Int :: *
data Int = I# Int#
  	-- Defined in ‘GHC.Types’
instance Bounded Int -- Defined in ‘GHC.Enum’
instance Enum Int -- Defined in ‘GHC.Enum’
instance Integral Int -- Defined in ‘GHC.Real’
instance Num Int -- Defined in ‘GHC.Num’
instance Read Int -- Defined in ‘GHC.Read’
instance Real Int -- Defined in ‘GHC.Real’
instance Ord Int -- Defined in ‘GHC.Classes’
instance Eq Int -- Defined in ‘GHC.Classes’
instance Show Int -- Defined in ‘GHC.Show’

type Integer :: *
data Integer = IS Int# | IP ByteArray# | IN ByteArray#
  	-- Defined in ‘GHC.Num.Integer’
instance Enum Integer -- Defined in ‘GHC.Enum’
instance Integral Integer -- Defined in ‘GHC.Real’
instance Num Integer -- Defined in ‘GHC.Num’
instance Read Integer -- Defined in ‘GHC.Read’
instance Real Integer -- Defined in ‘GHC.Real’
instance Ord Integer -- Defined in ‘GHC.Num.Integer’
instance Eq Integer -- Defined in ‘GHC.Num.Integer’
instance Show Integer -- Defined in ‘GHC.Show’

Podobná pohádka to je i u desetinných čísel. Pracujeme se dvěma základními typy `Double` a `Float`, ale jejich zobecněním je typová třída `Fractional`.

In [57]:
3.14 :: Float
3.14 :: Double
:t 3.14
:t 1/2

3.14

3.14

3.14 :: forall {a}. Fractional a => a

1/2 :: forall {a}. Fractional a => a

In [58]:
:i Fractional

type Fractional :: * -> Constraint
class Num a => Fractional a where
  (/) :: a -> a -> a
  recip :: a -> a
  fromRational :: Rational -> a
  {-# MINIMAL fromRational, (recip | (/)) #-}
  	-- Defined in ‘GHC.Real’
instance Fractional Double -- Defined in ‘GHC.Float’
instance Fractional Float -- Defined in ‘GHC.Float’

Občas nás to přivede do zdánlivě zvláštních situací. Například tohle vypadá jako dělení dvou celých čísel, ale lze to v pohodě vyhodnotit:

In [59]:
4 / 3

1.3333333333333333

Tohle ale skončí chybou:

In [60]:
someList = [1, 2, 3]
4 / length someList

: 

**Otázka:** V čem je problém? (Hint: použijte `:t` na operátor i na funkci.)

Jen na okraj: Haskell dokonce obsahuje i typ pro zlomky, pomocí kterého se dají přesně vyjadřovat racionální čísla.

In [61]:
3.14 :: Rational
(1/2::Rational) / (3/4)
-- srovnejte s:
(1/2) / (3/4)

157 % 50

2 % 3

0.6666666666666666

In [62]:
:i Rational
import Data.Ratio
:i Ratio

type Rational :: *
type Rational = Ratio Integer
  	-- Defined in ‘GHC.Real’

type Ratio :: * -> *
data Ratio a = !a :% !a
  	-- Defined in ‘GHC.Real’
instance Integral a => Enum (Ratio a) -- Defined in ‘GHC.Real’
instance Integral a => Fractional (Ratio a) -- Defined in ‘GHC.Real’
instance Integral a => Num (Ratio a) -- Defined in ‘GHC.Real’
instance (Integral a, Read a) => Read (Ratio a) -- Defined in ‘GHC.Read’
instance Integral a => Real (Ratio a) -- Defined in ‘GHC.Real’
instance Integral a => RealFrac (Ratio a) -- Defined in ‘GHC.Real’
instance Integral a => Ord (Ratio a) -- Defined in ‘GHC.Real’
instance Show a => Show (Ratio a) -- Defined in ‘GHC.Real’
instance Eq a => Eq (Ratio a) -- Defined in ‘GHC.Real’

`minBound` a `maxBound` jsou „polymorfní konstanty“ – funkce, které nemají žádný vstup, ale čistě podle svého typu nabydou nějaké konstantní hodnoty.

In [63]:
:t minBound

minBound :: forall a. Bounded a => a

Pokud je chceme použít pro zjištění spodní/horní meze nějakého typu, musíme explicitně říct, o jakém typu se bavíme:

In [64]:
(minBound :: Int)
(maxBound :: Int)

-9223372036854775808

9223372036854775807

Bez něj bude výsledkem jen „dummy hodnota“:

In [65]:
minBound

()

**Otázka:** Co bude výsledkem následujícího výrazu?

In [66]:
['a', minBound]

"a\NUL"

### Porovnávání

In [67]:
x = 10
x == 10
x > 6
x < 2
x /= 10

True

True

False

False

Asi čekáte, že u porovnávacích operátorů se opět uplatní touha abstrahovat a nebudou tedy definovány jen na číslech:

In [68]:
:t (<)
:t (<=)

(<) :: forall a. Ord a => a -> a -> Bool

(<=) :: forall a. Ord a => a -> a -> Bool

Tyhle typové signatury říkají `<` je funkce, která pracuje s libovolným typem `a`, který je instancí `Ord`. Jedním z takových typů je třeba i `Bool`:

In [69]:
:i Bool

type Bool :: *
data Bool = False | True
  	-- Defined in ‘GHC.Types’
instance Bounded Bool -- Defined in ‘GHC.Enum’
instance Enum Bool -- Defined in ‘GHC.Enum’
instance Read Bool -- Defined in ‘GHC.Read’
instance Ord Bool -- Defined in ‘GHC.Classes’
instance Show Bool -- Defined in ‘GHC.Show’
instance Eq Bool -- Defined in ‘GHC.Classes’

In [70]:
True > False

True

Náš `MyBool` ale `Ord` neimplementuje, takže tady máme smůlu:

In [71]:
:i MyBool

type MyBool :: *
data MyBool = MyFalse | MyTrue
  	-- Defined at <interactive>:1:1
instance [safe] Show MyBool -- Defined at <interactive>:2:14

In [72]:
MyFalse > MyTrue

: 

`Ord` implementují i seznamy:

In [73]:
[1, 2] > [2, 1]

False

In [74]:
[2, 3, 1] < [4, 5]

True

In [75]:
[2, 3, 1] < [2, 3]

False

Protože některé věci mohou jít testovat na rovnost, ale není mezi nimi uspořádání, existuje taky typová třída `Eq`, která umožňuje jen používat `==` a `/=`. Příkladem takové věci jsou třeba komplexní čísla:

In [76]:
import Data.Complex
:i Complex

type Complex :: * -> *
data Complex a = !a :+ !a
  	-- Defined in ‘Data.Complex’
instance Foldable Complex -- Defined in ‘Data.Complex’
instance Traversable Complex -- Defined in ‘Data.Complex’
instance RealFloat a => Floating (Complex a) -- Defined in ‘Data.Complex’
instance RealFloat a => Fractional (Complex a) -- Defined in ‘Data.Complex’
instance RealFloat a => Num (Complex a) -- Defined in ‘Data.Complex’
instance Read a => Read (Complex a) -- Defined in ‘Data.Complex’
instance Applicative Complex -- Defined in ‘Data.Complex’
instance Functor Complex -- Defined in ‘Data.Complex’
instance Monad Complex -- Defined in ‘Data.Complex’
instance Show a => Show (Complex a) -- Defined in ‘Data.Complex’
instance Eq a => Eq (Complex a) -- Defined in ‘Data.Complex’

In [77]:
(3 :+ 1) == (3 :+ 1)

True

**Otázka:** `elem :: (Eq a) => a -> [a] -> Bool` \
Odhadněte podle typové signatury, co tato funkce `elem` dělá.

### if, then, else
I ve světě vyhodnocování funkcí dává smysl konstrukce pokud něco – pak použij tento výraz – jinak použij tento výraz:

In [78]:
if True then "gde" else "body"

"gde"

Je dobré si uvědomit, že to celé je zase jenom výraz, který „vrací“ hodnotu (lépe „vyhodnotí se“). To taky znamená, že část *then* i část *else* musí být přítomny a musí být stejného typu.

Mohli bychom to celé obalit do funkce:

In [79]:
ifFun :: Bool -> a -> a -> a
ifFun cond a b = if cond then a else b

In [80]:
ifFun True "gde" "body"
ifFun False "gde" "body"

"gde"

"body"

**Otázka:** Co se stane, když v typové signatuře funkce `ifFun` změním `Bool` za `a`?

**Čas vypracovat příklady 3 až 5 z odevzdávaného notebooku.**